In [1]:
import torch
import pandas as pd
import os
from rinalmo.pretrained import get_pretrained_model
DEVICE = "cuda:0"
import numpy as np
np.random.seed(42)

In [2]:
model, alphabet = get_pretrained_model(model_name="giga-v1")
model = model.to(device=DEVICE)
model.eval()

RiNALMo(
  (embedding): Embedding(22, 1280, padding_idx=1)
  (transformer): Transformer(
    (blocks): ModuleList(
      (0-32): 33 x TransformerBlock(
        (mh_attn): FlashMultiHeadSelfAttention(
          (rotary_emb): RotaryEmbedding()
          (flash_self_attn): FlashAttention()
          (Wqkv): Linear(in_features=1280, out_features=3840, bias=False)
          (attention_dropout): Dropout(p=0.1, inplace=False)
          (out_proj): Linear(in_features=1280, out_features=1280, bias=False)
        )
        (attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (transition): Sequential(
          (0): SwiGLU(
            (linear): Linear(in_features=1280, out_features=3413, bias=True)
            (linear_gate): Linear(in_features=1280, out_features=3413, bias=True)
          )
          (1): Dropout(p=0.0, inplace=False)
          (2): Linear(in_features=3413, out_features=1280, bias=True)
        )
        (out_layer_norm): LayerNorm((1280,), eps=1e-05

In [3]:
rfam_ids = pd.read_csv("sequence_rfam_mapping_annotations.csv") #their corresponding rfam ids (family)
alignment_score_matrix = pd.read_csv("alignment_matrix_3_perfam.csv", index_col=0) #pairwise alignment matrix aren shared with me
ids = alignment_score_matrix.index.tolist() #get indices which are in row and column names 
fasta_df = pd.read_csv("fasta_df.csv") #contains sequence_id, sequence, and length for every sequence
fasta_df = fasta_df.iloc[ids] #only keep the ones from ids
rfam_ids = rfam_ids.iloc[ids] #only keep the ones from ids

In [34]:
eukaryotic_indexes = rfam_ids.index[rfam_ids["rfam_id"] == "RF01960"]
eukarytoic_values = fasta_df.loc[eukaryotic_indexes, "sequence"].tolist()

In [51]:
original = eukarytoic_values[2]
seqs = original
seqs

'CTGGTTGATCCTCCTAGTAGTATATGCTTGACTCAAAGATTAGGCCATGCAAGTCTAAGTACAATTGACTAGTACAGTGAAACTGTGAATGGCTCATCAGTTATGGTTCCTTGATCGCTCCAATGTTACCTAAATAACTGTGGCAATTTTAGAGCTAATACATGTTAATGAGCATTGACCTTTGGGGATGGGTGCATTTATCAAACCCAAAACCCATGTGGGGTACTATCGGGTGTCGCTTTTGGTGAGTCTAGATGACCTCGAGCCATTTGTGCGTCCTTTGTGGCGGTGATGTCTCATTCGAATGTATGCCCTATCAACTTTCAACGGTACTCTCTGTGTCTACCATGGTGACCACAGGTAATAGGGATTCAGGGTTTGATCCTGGAGAAGGAGCCTAAGAAACGGCTACCGTACACAAGGATTAAGGCAGCAGGCGCACAAATTACCCACTGCCAACTCTGGGAGGTATACCAAAAATAACAAAACAGGACTCTCGAGGCCCTGTAATTGGAATGAGTACACATTAAATTATTTGACAAGGATCCATTGCTAATGTTTAAAAGTTGCATTTT'

In [52]:
for b in range(1,11):
    slider = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
    while True:
        tokens = torch.tensor(alphabet.batch_tokenize([seqs]), dtype=torch.int64, device=DEVICE) # FIXED
        slider_checked = [x for x in slider if x in range(1,len(seqs)+1)]
        if slider_checked:
            masked_tokens = tokens.clone()
            masked_tokens[0, slider_checked] = alphabet.mask_idx
            with torch.no_grad(), torch.cuda.amp.autocast():
                outputs = model(masked_tokens)
            logits = outputs["logits"]
            mask_logits = logits[0, slider_checked]
            predicted_token_ids = torch.argmax(mask_logits, dim=-1)
            predicted_nucleotides = [alphabet.idx_to_tkn[i.item()] for i in predicted_token_ids]
            seqs_list = list(seqs)
            for pos, nucleotide in zip(slider_checked, predicted_nucleotides):
                seqs_list[pos-1] = nucleotide
            seqs = "".join(seqs_list)
            slider = [x + 1 for x in slider]
        else:  
            break    

In [53]:
seqs

'TGGTTGATCCTGCCAGTAGTCATATGCTTGTCTCAAAGATTAAGCCATGCATGTCTAAGTATAAACTGCTTTAACTGTGAAACTGCGAATGGCTCATTAAATCAGTTATAGTTTATTTGATGGTACCTACTACTTGGATAACCGTAGTAATTCTAGAGCTAATACATGCGAAAAACCCCGACTTCTGGAAGGGGTGTATTTATTAGATTAAAAACCAATGCGGGTCTTCGGGCCCGTTTTTGTTGGTGATTCATAATAACTTTTCGAATCGCATGGCCTTGTGCCGGCGATGCTTCATTCAAATTTCTGCCCTATCAACTTTCGATGGTAGGATAGAGGCCTACCATGGTTTTTAACGGGTAACGGGGAATTAGGGTTCGATTCCGGAGAGGGAGCCTGAGAAACGGCTACCACATCCAAGGAAGGCAGCAGGCGCGCAAATTACCCAATCCCGACACGGGGAGGTAGTGACAATAAATAACAATACAGGGCTCTTTTGGGTCTTGTAATTGGAATGAGTACAATTTAAATCCCTTAACGAGGAACAATTGGAGGGCAAGTCTGGTGCCAGCAGC'

In [54]:
mismatches = []
for i, (orig, cons) in enumerate(zip(original, seqs)):
    if orig != cons:
        mismatches.append({
            'position': i + 1, # +1 for 1-based biological indexing
            'original': orig,
            'consensus': cons
        })
mismatches

[{'position': 1, 'original': 'C', 'consensus': 'T'},
 {'position': 2, 'original': 'T', 'consensus': 'G'},
 {'position': 4, 'original': 'G', 'consensus': 'T'},
 {'position': 6, 'original': 'T', 'consensus': 'G'},
 {'position': 7, 'original': 'G', 'consensus': 'A'},
 {'position': 8, 'original': 'A', 'consensus': 'T'},
 {'position': 9, 'original': 'T', 'consensus': 'C'},
 {'position': 11, 'original': 'C', 'consensus': 'T'},
 {'position': 12, 'original': 'T', 'consensus': 'G'},
 {'position': 15, 'original': 'T', 'consensus': 'A'},
 {'position': 16, 'original': 'A', 'consensus': 'G'},
 {'position': 17, 'original': 'G', 'consensus': 'T'},
 {'position': 18, 'original': 'T', 'consensus': 'A'},
 {'position': 19, 'original': 'A', 'consensus': 'G'},
 {'position': 20, 'original': 'G', 'consensus': 'T'},
 {'position': 21, 'original': 'T', 'consensus': 'C'},
 {'position': 31, 'original': 'A', 'consensus': 'T'},
 {'position': 43, 'original': 'G', 'consensus': 'A'},
 {'position': 52, 'original': 'A', 

In [55]:
#I will make some sequences up same length as the above 3
for i in eukarytoic_values: print(len(i))
sequence_first = "".join(np.random.choice(list("AUCG"), size=763))
sequence_second = "".join(np.random.choice(list("AUCG"), size=781))
sequence_third = "".join(np.random.choice(list("AUCG"), size=575))

763
781
575


In [56]:
original = sequence_first
seqs = original
seqs

'CGACCGAACUCCCCGAGGGCUAUGGUUUGGAAGUUAGAACCCUGGGGCUUCUCGCGGACACCAACUGAGUUUAUAUGGCGCGAGCCUAGUGGUUUUUGUACUUGUUUGUCGCGUCGAUGAGAUCAGUAGGGAAACAAACAGAGGGCCCAGCCACAUCUAGCAGGUAGCCUGACGGUCCACACUCAAUCCUCCACCUUGACCGCAGAGGUACCACCAGAGCCCUGUUAUAAUGGGGGUUCGUCGACUAAACUAGAACCUGCAUAACUGGCCUGGGAGAUAUGGUCUCAAAGACAUUGUCAGAACUUAGUGUGCGCCGCACUGAGUUUCCGGAAGCUGACGGCGCUCCCGGCGAAGGCUGACGAAUCCUUCCGGUGAGGAUAGUGAAGCCAACCGGCGGUGAGGCAGGUGGUCGUACAAUGUUUUCGAAGAGAUAGGGGGCCAGAGGCCUCUUUUUACUGCCUAUAGCGAAGAGCGCGAGAGGUAUAUCGAAGAAUACCGAGCAAGCAGCGUAUCUUCGUGUGCUCUCCUUUAGAACUGCAUCUCUAGAGUCAGAGAGGUGUGCCCGUUGCCUCAUCUAUUUUAGUUCUCGAAGAGGGACGUAGGUCCCGGGAUGUUCAGCCGUCUAUAUCCAGAAUUACUGUUGAGAAAGACGCAUACCGUGCAUAAUGACAUUUGAGGUUGACGAUUACUUUAGUGCUUCUGAGAAGGGGCUAGGGCGGGUCGAAACCAGCAAGUCGAAGCUUACUCCAUGAUUUUUCACGUA'

In [57]:
for b in range(1,11):
    slider = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
    while True:
        tokens = torch.tensor(alphabet.batch_tokenize([seqs]), dtype=torch.int64, device=DEVICE) # FIXED
        slider_checked = [x for x in slider if x in range(1,len(seqs)+1)]
        if slider_checked:
            masked_tokens = tokens.clone()
            masked_tokens[0, slider_checked] = alphabet.mask_idx
            with torch.no_grad(), torch.cuda.amp.autocast():
                outputs = model(masked_tokens)
            logits = outputs["logits"]
            mask_logits = logits[0, slider_checked]
            predicted_token_ids = torch.argmax(mask_logits, dim=-1)
            predicted_nucleotides = [alphabet.idx_to_tkn[i.item()] for i in predicted_token_ids]
            seqs_list = list(seqs)
            for pos, nucleotide in zip(slider_checked, predicted_nucleotides):
                seqs_list[pos-1] = nucleotide
            seqs = "".join(seqs_list)
            slider = [x + 1 for x in slider]
        else:  
            break    

In [58]:
seqs

'TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT'

In [59]:
mismatches = []
for i, (orig, cons) in enumerate(zip(original, seqs)):
    if orig != cons:
        mismatches.append({
            'position': i + 1, # +1 for 1-based biological indexing
            'original': orig,
            'consensus': cons
        })
mismatches

[{'position': 1, 'original': 'C', 'consensus': 'T'},
 {'position': 2, 'original': 'G', 'consensus': 'T'},
 {'position': 3, 'original': 'A', 'consensus': 'T'},
 {'position': 4, 'original': 'C', 'consensus': 'T'},
 {'position': 5, 'original': 'C', 'consensus': 'T'},
 {'position': 6, 'original': 'G', 'consensus': 'T'},
 {'position': 7, 'original': 'A', 'consensus': 'T'},
 {'position': 8, 'original': 'A', 'consensus': 'T'},
 {'position': 9, 'original': 'C', 'consensus': 'T'},
 {'position': 10, 'original': 'U', 'consensus': 'T'},
 {'position': 11, 'original': 'C', 'consensus': 'T'},
 {'position': 12, 'original': 'C', 'consensus': 'T'},
 {'position': 13, 'original': 'C', 'consensus': 'T'},
 {'position': 14, 'original': 'C', 'consensus': 'T'},
 {'position': 15, 'original': 'G', 'consensus': 'T'},
 {'position': 16, 'original': 'A', 'consensus': 'T'},
 {'position': 17, 'original': 'G', 'consensus': 'T'},
 {'position': 18, 'original': 'G', 'consensus': 'T'},
 {'position': 19, 'original': 'G', 'c